In [ ]:
%%html
<style>
toc {
    position: fixed;
    top: 50px;
    left: 20px;
    width: 250px;
    max-height: 80vh;
    overflow-y: auto;
    background: #f8f8f8;
    padding: 15px;
    border: 1px solid #ddd;
    border-radius: 5px;
    z-index: 1000;
}

</style>

<script>
// Auto-generate TOC from headers
document.addEventListener("DOMContentLoaded", function() {
    const headers = document.querySelectorAll('h1, h2, h3');
    const toc = document.getElementById('toc');
    
    if (toc && headers.length > 0) {
        let tocHTML = '<h3>Contents</h3><ul>';
        headers.forEach((header, index) => {
            const id = 'section-' + index;
            header.id = id;
            const level = parseInt(header.tagName[1]);
            const indent = (level - 1) * 20;
            tocHTML += `<li style="margin-left: ${indent}px">
                <a href="#${id}">${header.textContent}</a>
            </li>`;
        });
        tocHTML += '</ul>';
        toc.innerHTML = tocHTML;
    }
});
</script>

# Table of Contents
<div id="toc"></div>

Setting up parameter block for papermill

In [ ]:
# parameter defaults for testing - overwritten by papermill
SUBSET_MODE = "DICT"

FILE_NAME = "Plasma_Cells_scRNA"
ADDITIONAL_SUBDIR = "Plasma_Cells/"
ANALYSIS_DESC = "ULM Scoring across scRNA-seq Plasma Cells, per Sample"
COMPARTMENT_DESCRIPTION = "scRNA Plasma Cells"

RELOAD_IF_EXISTING=True
# Example contents
#cellID_short=dict(include=["NK_adp","NK_CD56dim","NK_CD56bright","NK_resident"],exclude=None), 
#lineage_group=dict(include=None, exclude=["LQ"])

# Legacy parameter
CLUSTERS_ANALYSIS = ["NK_adp"]

CUSTOM_PATHWAYS = {
    "Plasma_cell": ["SDC1","JCHAIN","MZB1","DERL3","IGKC","IGLC2","IGLC3","IGLC6","IGLC7"],
}
SUBSET_MODE_CUSTOM_DICT = dict()


Setting up output covariates and other things

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import decoupler as dc
from ipynbcompress import compress
import ipynbname
from datetime import datetime
import os

print(os.getcwd())
os.chdir('/workspaces/Junia_Collab_Spatial/')
print(os.getcwd())


current_time = datetime.today().strftime("%Y-%m-%d;%H:%M:%S")

print(f'Notebook Run Time: {current_time}')
print(f'Container Version: {os.environ.get("CONTAINER_VERSION")}')
print(f'Compartment Description: {COMPARTMENT_DESCRIPTION}')
print(f'Analysis Description: {ANALYSIS_DESC}')

sub_directory = f'{ADDITIONAL_SUBDIR}/{FILE_NAME}/' 

# Inspect Object
output_dir_data_large = "data/"
output_dir_fig_tables = "output/Junia_Spatial_Analyses/" + sub_directory

os.makedirs(output_dir_fig_tables, exist_ok=True)

Author: William Pilcher, Emory University/Georgia Institute of Technology

This performs the following:
    1) Loads .h5ad data containing converted Seurat objects with counts in raw.X, and log-normalized data (cp10k) in .X
    2) Subsets to Baseline + the Cell Type of Interest
    3) Performs pseudobulking via the decoupler package, filtering rare genes and patients with fewer than 10 cells of the compartment of interest
    4) py-deseq to perform differential expression for AFR Ancestry
    5) py-deseq for LFC shrinkage for GSEA from hongwei
    6) Tests on DEG results with decoupler implementations of the target/receptor network, progeny, and hallmark.
    7) (After manually saving the notebook), exports notebook to .html

As part of an additional screening step, we will also quickly assess the outcomes of all markers just to see if any relationship exists between the strongly ancestry associated markers and patient outcomes. These do not necessarily reflect the final statistics for reporting survival.

Only the differentially expressed genes will be the 'final' result generated by this report. Pathway analysis will be subject to secondary assessment via GSEA. 

Final visualizations will be done in R. 

Printing time notebook was generated along with the container image version.

Note - the container image will only contain python analysis package and will not include notebook functionality or things like 'ipynbcompress'. These are installed in the devcontainer dockerfile.

<h1>Loading and initial preprocessing</h1>

In [ ]:
from datetime import datetime
import os


current_time = datetime.today().strftime("%Y-%m-%d;%H:%M:%S")

print(f'Notebook Run Time: {current_time}')
print(f'Container Version: {os.environ.get("CONTAINER_VERSION")}')


Loading the full object, and checking file contents

In [ ]:
# This is the large object
filename = output_dir_data_large + 'h5ad/Combined_Full_Object/Combined_Full_Object.h5ad'
adata = sc.read_h5ad(filename=filename, chunk_size=20000)


In [ ]:
# Just checking embeddings
adata.obsm

In [ ]:
adata.obs.head()

Ensuring pandas is treating the continuous covariate columns we will be using as numeric covariates

In [ ]:
# Post conversion - numeric covariates treated as factors. Ensure they are correctly treated as numeric for downstream analyses.
cont_cov_col = ['d_dx_amm_age','d_dx_amm_bmi','ttcpfs','ttcos']

adata.obs[cont_cov_col] = adata.obs[cont_cov_col].apply(pd.to_numeric, errors='coerce')
adata.obs.dtypes

Subset to baseline + the cell types of interest

In [ ]:
#papermill_description=SUBSETTING_LONG

# Subset to baseline samples in the MMRF object.
# Removing .copy - memory usage oddities?
import gc

adata = adata[adata.obs['VJ_INTERVAL'] == 'Baseline']
if(SUBSET_MODE == "CLUSTER"):
    print(CLUSTERS_ANALYSIS)
    adata = adata[adata.obs['cellID_short'].isin(CLUSTERS_ANALYSIS)]
    adata.obs['cellID_short'].value_counts()


elif(SUBSET_MODE == "DICT"):
    for(key, val) in SUBSET_MODE_CUSTOM_DICT.items():
        print(f'Subsetting {key}')
        include_list = val.get('include')
        exclude_list = val.get('exclude')
        if(include_list is not None):
            adata = adata[adata.obs[key].isin(include_list)]
        if(exclude_list is not None):
            adata = adata[~adata.obs[key].isin(exclude_list)]
        print(adata.obs[key].value_counts())
    for(key, val) in SUBSET_MODE_CUSTOM_DICT.items():
        print(f'Final counts for {key}:')
        print(adata.obs[key].value_counts())
        
gc.collect() # Force garbage collection to free memory - can sometimes be an issue with large AnnData objects


Creating other covariates and moving counts from raw -> layers

In [ ]:
# Seurat creates a .raw slot instead of using .layers for counts data. Copy the raw matrix to a layer, and delete the raw slot.
adata.layers['counts'] = adata.raw.X.copy()
adata.raw = None

<h1> Decoupler Pseudobulking </h1>

In [ ]:
#papermill_description=Pseudobulking
pdata = dc.pp.pseudobulk(
    adata=adata,
    sample_col="public_id",
    groups_col=None,
    mode="sum",
    layer='counts'
)
dc.pl.filter_samples(adata=pdata, min_cells=10, min_counts=1000)
dc.pl.obsbar(adata=pdata, y="Study_Site", figsize=(6, 3))

# We don't use the single-cell object past this point - just delete it
del adata 
gc.collect() # Force garbage collection to free memory immediately



In [ ]:
# Ensuring the covariates are correctly numeric in the pseudobulked object
cont_cov_col = ['d_dx_amm_age','d_dx_amm_bmi','ttcpfs','ttcos']

pdata.obs[cont_cov_col] = pdata.obs[cont_cov_col].apply(pd.to_numeric, errors='coerce')
pdata.obs.dtypes

Sample filtering 

In [ ]:
dc.pp.filter_samples(adata=pdata, min_cells=10, min_counts=1000)

In [ ]:
dc.pl.obsbar(adata=pdata, y="Study_Site", figsize=(6, 3))


Assessing for batch effect/variance in pseudobulked PCs

In [ ]:
# Store raw counts in layers
pdata.layers["counts"] = pdata.X.copy()

# Normalize, scale and compute pca
sc.pp.normalize_total(pdata, target_sum=1e4)
pdata.layers["nolog_normalized"] = pdata.X.copy()
sc.pp.log1p(pdata)
pdata.layers["normalized"] = pdata.X.copy()
sc.pp.scale(pdata, max_value=10)
pdata.layers["scaled"] = pdata.X.copy()

sc.tl.pca(pdata)

# Return raw counts to X
dc.pp.swap_layer(adata=pdata, key="counts", inplace=True)

In [ ]:
pdata.obs.head()

In [ ]:
pdata.obsm.keys()

In [ ]:
dc.tl.rankby_obsm(pdata, key="X_pca", obs_keys=["Study_Site","siteXbatch","Batch","d_pt_sex","d_dx_amm_age", "IMWG_2024_HR", "psbulk_cells", 'psbulk_counts'])

In [ ]:
if(FILE_NAME!="Fibro"):
    sc.pl.pca_variance_ratio(pdata)
    dc.pl.obsm(adata=pdata, return_fig=True, nvar=7, titles=["PC scores", "Adjusted p-values"], figsize=(10, 5))
else:
    print("Err in Fibro class- skipping QC plots")

In [ ]:
sc.pl.pca(
    pdata,
    color=["psbulk_cells", 'psbulk_counts', 'd_pt_sex', 'Study_Site', 'siteXbatch', 'IMWG_2024_HR', 'd_dx_amm_age'],
    ncols=2,
    size=300,
    frameon=True,
)

Filtering by gene expression:

In [ ]:
dc.pl.filter_by_expr(
    adata=pdata,
    min_count=5,
    min_total_count=7,
    large_n=5,
    min_prop=0.1,
)


In [ ]:
dc.pp.filter_by_expr(
    adata=pdata,
    min_count=5,
    min_total_count=7,
    large_n=5,
    min_prop=0.1,
)


In [ ]:
pdata_filt = pdata # Don't need to copy 

<h3> Custom Gene Sets </h3>

In [ ]:
#papermill_description=Decoupler_Custom

# Convert CUSTOM_PATHWAYS dictionary to pandas DataFrame
# Each row is a gene-pathway pair
pathway_data = []
for pathway_name, genes in CUSTOM_PATHWAYS.items():
    for gene in genes:
        pathway_data.append({'source': pathway_name, 'target': gene})

custom_pathways_df = pd.DataFrame(pathway_data)
print(f"Created pathway dataframe with {len(custom_pathways_df)} gene-pathway pairs")
print(f"Number of unique pathways: {custom_pathways_df['source'].nunique()}")
print(f"Number of unique genes: {custom_pathways_df['target'].nunique()}")
custom_pathways_df = custom_pathways_df.drop_duplicates(subset=['source','target'])
custom_pathways_df.head()

ULM Based Approach

In [ ]:
dc.mt.ulm(data=pdata_filt, net=custom_pathways_df, layer='normalized')

In [ ]:
pdata_filt.obsm['score_ulm']

In [ ]:
print(f"Saved custom pathways ULM scores to: {output_dir_fig_tables}{FILE_NAME}_custom_pathways_ulm_scores.csv")

custom_ulm_scores = pd.DataFrame(pdata_filt.obsm['score_ulm'], index=pdata_filt.obs_names)
custom_ulm_scores.to_csv(output_dir_fig_tables + f"{FILE_NAME}_custom_pathways_ulm_scores.csv")

<h1> Gene expression survival analyses </h1>

In [ ]:
#papermill_description=Survival

import sys
n_pat = pdata_filt.n_obs
if(n_pat < 23):
    print(f"Too few patient for survival analyses")
    sys.exit(0)
else:
    print(f"Survival analyses can continue")


In [ ]:
# Convert survival columns to float to avoid categorical issues
pdata_filt.obs['ttcpfs'] = pd.to_numeric(pdata.obs['ttcpfs'], errors='coerce')
pdata_filt.obs['censpfs'] = pd.to_numeric(pdata.obs['censpfs'], errors='coerce')

print("Converted survival columns to numeric:")
print(f"ttcpfs dtype: {pdata_filt.obs['ttcpfs'].dtype}")
print(f"censpfs dtype: {pdata_filt.obs['censpfs'].dtype}")
print(f"ttcpfs range: {pdata_filt.obs['ttcpfs'].min():.1f} - {pdata_filt.obs['ttcpfs'].max():.1f}")
print(f"censpfs values: {pdata_filt.obs['censpfs'].unique()}")

In [ ]:
#papermill_description=COX_model_study_site

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_site_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_study_site_only_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(cox_site_results_file):
    print(f"Loading existing Cox regression (Study_Site model) results from: {cox_site_results_file}")
    test_results_site = pd.read_csv(cox_site_results_file, index_col=0)
    print(f"Loaded {len(test_results_site)} genes from existing results")
else:
    print("=== Running Cox regression analysis with all genes (Study_Site adjustment) ===")
    test_results_site = wp_scanpy.genome_wide_cox_analysis(
        pdata_filt, 
        layer='scaled',
        covariates=['Study_Site'],
        max_genes=None,
        verbose=True
    )
    
    test_results_site["model"] = "Study_Site"
    test_results_site["survival_type"] = "PFS"
    test_results_site.to_csv(cox_site_results_file)

# Display results
print("Top 10 results (by p-value):")
top_results = test_results_site[test_results_site['status'] == 'success'].nsmallest(30, 'p_value')
print(top_results[['gene', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

if (test_results_site['p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_site, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Study_Site Adjusted) - Raw P-Values",
                    p_val_column='p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at p < 0.05 - Skipping Plot")

if (test_results_site['fdr_p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_site, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Study_Site Adjusted) - FDR Corrected",
                    p_val_column='fdr_p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at FDR padj < 0.05 - Skipping Plot")


In [ ]:
#papermill_description=COX_model_full

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
cox_full_results_file = output_dir_fig_tables + f"{FILE_NAME}_cox_regression_full_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(cox_full_results_file):
    print(f"Loading existing Cox regression (full model) results from: {cox_full_results_file}")
    test_results_full = pd.read_csv(cox_full_results_file, index_col=0)
    print(f"Loaded {len(test_results_full)} genes from existing results")
else:
    print("=== Running Cox regression analysis with all genes (Full Model) ===")
    test_results_full = wp_scanpy.genome_wide_cox_analysis(
        pdata_filt, 
        layer='scaled',
        covariates=['d_dx_amm_age','d_amm_tx_asct_1st', 'd_pt_sex', 'Study_Site'],
        max_genes=None,
        verbose=True
    )
    
    test_results_full["model"] = "d_dx_amm_age + d_amm_tx_asct_1st + d_pt_sex + Study_Site"
    test_results_full["survival_type"] = "PFS"
    test_results_full.to_csv(cox_full_results_file)

# Display results
print("Top 10 results (by p-value):")
top_results = test_results_full[test_results_full['status'] == 'success'].nsmallest(30, 'p_value')
print(top_results[['gene', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

if (test_results_full['p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_full, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Full Model) - Raw P-Values",
                    p_val_column='p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at p < 0.05 - Skipping Plot")

if (test_results_full['fdr_p_value'] < 0.05).any():
    wp_scanpy.plot_cox_volcano(test_results_full, 
                    title=f"{COMPARTMENT_DESCRIPTION}: Cox Regression: Gene Expression vs Survival (Full Model) - FDR Corrected",
                    p_val_column='fdr_p_value',
                    p_threshold=0.05, 
                    top_n_labels=75)
else:
    print("No significant genes found at FDR padj < 0.05 - Skipping Plot")


Test for Outcome Associations

<h3> Outcome Associated Pathways via Decoupler </h3>

Repeating the decoupler analyses, but basically doing it by sign(log(HR)) * -log(p-val) from the adjusted coxPH model.

In [ ]:
test_results_full = test_results_full.set_index('gene')
test_results_full

In [ ]:
test_results_full[["hazard_ratio_log10"]] = test_results_full[["hazard_ratio"]].apply(np.log2)

test_results_full["signed_pval"] = test_results_full["p_value"].apply(np.log2).multiply(-1).multiply(test_results_full["hazard_ratio_log10"].apply(np.sign))


data = test_results_full[["p_value"]].T.rename(index={"p_value": "AFRHigh_vs_AFRLow"}).apply(np.log2).multiply(-1)
data

dataHR = test_results_full[["hazard_ratio"]].T.rename(index={"hazard_ratio": "AFRHigh_vs_AFRLow"}).apply(np.log2)
dataHR

test_results_full


In [ ]:
data_mult = data.multiply(dataHR.apply(np.sign))

data_mult

<h3> CoxPH - Custom Pathways </h3>

In [ ]:
# Run
custom_acts, custom_padj = dc.mt.ulm(data=data_mult, net=custom_pathways_df)

# Filter by sign padj
msk = (custom_padj.T < 0.05).iloc[:, 0]
custom_acts = custom_acts.loc[:, msk]

custom_acts

In [ ]:

if(len(custom_acts.columns) > 0):
    dc.pl.barplot(data=custom_acts, name="AFRHigh_vs_AFRLow", figsize=(6, 3))
else:
    print("No significant hallmark pathways found")

if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        _, le = dc.pl.leading_edge(
            test_results_full,
            stat="signed_pval",
            net=custom_pathways_df,
            name=pathway,
        )
        print(f"{pathway} leading edge:", le[:10])
else:
    print("No significant custom pathways found")


if(len(custom_acts.columns) > 0):
    for pathway in custom_acts.columns.tolist():
        dc.pl.volcano(
            data=test_results_full,
            x="hazard_ratio_log10",
            y="p_value",
            net=custom_pathways_df,
            name=pathway,
            top=10,
            thr_stat=0.08,
            thr_sign=0.05
        )
else:
    print("No significant custom pathways found")

<h1>ULM Pathway Survival Analysis</h1>

Perform comprehensive survival analysis on all ULM pathway scores:
1. Cox regression with z-scored pathway scores
2. Forest plots visualizing hazard ratios
3. Kaplan-Meier curves with multiple stratification methods
4. Optimal cutpoint analysis with HR curves

## Cox Regression: Study Site Only

First analysis: adjusting only for Study_Site

In [ ]:
#papermill_description=ULM_COX_study_site

import importlib
from source.Helper_Functions import general_python as wp_scanpy
importlib.reload(wp_scanpy)

# Check if results already exist and should be reloaded
ulm_cox_site_results_file = output_dir_fig_tables + f"{FILE_NAME}_ulm_cox_regression_study_site_only_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(ulm_cox_site_results_file):
    print(f"Loading existing ULM Cox regression (Study_Site model) results from: {ulm_cox_site_results_file}")
    ulm_results_site = pd.read_csv(ulm_cox_site_results_file, index_col=0)
    print(f"Loaded {len(ulm_results_site)} pathways from existing results")
else:
    print("=== Running Cox regression analysis with all ULM pathways (Study_Site adjustment) ===")
    ulm_results_site = wp_scanpy.ulm_pathway_cox_analysis(
        pdata_filt, 
        ulm_obsm_key='score_ulm',
        covariates=['Study_Site'],
        verbose=True
    )
    
    ulm_results_site["model"] = "Study_Site"
    ulm_results_site["survival_type"] = "PFS"
    ulm_results_site.to_csv(ulm_cox_site_results_file)

# Display results
print("\nTop 10 pathways (by p-value):")
top_results = ulm_results_site[ulm_results_site['status'] == 'success'].nsmallest(10, 'p_value')
print(top_results[['pathway', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

In [ ]:
# Plot forest plot for Study_Site model
if (ulm_results_site['p_value'] < 0.05).any():
    wp_scanpy.plot_pathway_forest_plot(
        ulm_results_site,
        title=f"{COMPARTMENT_DESCRIPTION}: ULM Pathway Cox Regression (Study_Site Adjusted)",
        p_threshold=0.05,
        top_n=30,
        figsize=(10, 12)
    )
else:
    print("No significant pathways found at p < 0.05 - Skipping Forest Plot")

## Cox Regression: Fully Adjusted Model

Second analysis: adjusting for age, ASCT status, sex, and Study_Site

In [ ]:
#papermill_description=ULM_COX_fully_adjusted

# Check if results already exist and should be reloaded
ulm_cox_full_results_file = output_dir_fig_tables + f"{FILE_NAME}_ulm_cox_regression_fully_adjusted_model_results.csv"
if RELOAD_IF_EXISTING and os.path.exists(ulm_cox_full_results_file):
    print(f"Loading existing ULM Cox regression (Fully Adjusted model) results from: {ulm_cox_full_results_file}")
    ulm_results_full = pd.read_csv(ulm_cox_full_results_file, index_col=0)
    print(f"Loaded {len(ulm_results_full)} pathways from existing results")
else:
    print("=== Running Cox regression analysis with all ULM pathways (Fully Adjusted) ===")
    ulm_results_full = wp_scanpy.ulm_pathway_cox_analysis(
        pdata_filt, 
        ulm_obsm_key='score_ulm',
        covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex', 'Study_Site'],
        verbose=True
    )
    
    ulm_results_full["model"] = "Fully_Adjusted"
    ulm_results_full["survival_type"] = "PFS"
    ulm_results_full.to_csv(ulm_cox_full_results_file)

# Display results
print("\nTop 10 pathways (by p-value):")
top_results = ulm_results_full[ulm_results_full['status'] == 'success'].nsmallest(10, 'p_value')
print(top_results[['pathway', 'hazard_ratio', 'p_value', 'fdr_p_value', 'n_patients', 'n_events']])

In [ ]:
# Plot forest plot for Fully Adjusted model

wp_scanpy.plot_pathway_forest_plot(
    ulm_results_full,
    title=f"{COMPARTMENT_DESCRIPTION}: ULM Pathway Cox Regression (Fully Adjusted)",
    p_threshold=0.05,
    top_n=30,
    figsize=(10, 12)
)


## Combined Results

Merge results from both models into a single summary table

In [ ]:
# Combine results from both models into a wide format summary
ulm_combined_results = pd.merge(
    ulm_results_site[['pathway', 'hazard_ratio', 'hr_lower_ci', 'hr_upper_ci', 'p_value', 'fdr_p_value']],
    ulm_results_full[['pathway', 'hazard_ratio', 'hr_lower_ci', 'hr_upper_ci', 'p_value', 'fdr_p_value']],
    on='pathway',
    suffixes=('_site_only', '_fully_adjusted')
)

# Save combined results
combined_file = output_dir_fig_tables + f"{FILE_NAME}_ulm_cox_regression_combined_results.csv"
ulm_combined_results.to_csv(combined_file, index=False)
print(f"Saved combined ULM Cox regression results to: {combined_file}")

# Display top results from both models
print("\nTop 10 pathways by Study_Site model p-value:")
display(ulm_combined_results.nsmallest(10, 'p_value_site_only')[
    ['pathway', 'hazard_ratio_site_only', 'p_value_site_only', 
     'hazard_ratio_fully_adjusted', 'p_value_fully_adjusted']
])

print("\nTop 10 pathways by Fully Adjusted model p-value:")
display(ulm_combined_results.nsmallest(10, 'p_value_fully_adjusted')[
    ['pathway', 'hazard_ratio_site_only', 'p_value_site_only', 
     'hazard_ratio_fully_adjusted', 'p_value_fully_adjusted']
])

## Kaplan-Meier Curves for Top Pathways

For each significant pathway, generate KM curves with different stratification methods:
1. Median cutpoint
2. Quartile split (4 groups)
3. Optimal cutpoint (with HR curve showing all tested cutpoints)

In [ ]:
# Identify top pathways to plot (based on fully adjusted model)
# We'll plot KM curves for pathways with p < 0.05
significant_pathways = ulm_results_full[
    (ulm_results_full['status'] == 'success') & 
    (ulm_results_full['p_value'] < 0.05)
].sort_values('p_value')

print(f"Found {len(significant_pathways)} significant pathways (p < 0.05) to plot KM curves")
print("\nTop 10 significant pathways:")
print(significant_pathways.head(10)[['pathway', 'hazard_ratio', 'p_value']])

# Select top pathways to plot (limit to top 10 to avoid too many plots)
pathways_to_plot = ulm_results_full['pathway'].tolist() #significant_pathways.head(10)['pathway'].tolist()
print(f"\nWill generate KM curves for {len(pathways_to_plot)} pathways")

### KM Curves: Median Cutpoint

Split patients into high/low groups based on median pathway score

In [ ]:
# Generate KM curves with median cutpoint for each significant pathway
for pathway in pathways_to_plot:
    print(f"\n{'='*80}")
    print(f"Pathway: {pathway}")
    print('='*80)
    
    try:
        wp_scanpy.plot_pathway_km_curves(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            cutpoint_method='median',
            covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex', 'Study_Site'],
            figsize=(10, 6)
        )
    except Exception as e:
        print(f"Error plotting KM curve for {pathway}: {str(e)}")

### KM Curves: Quartile Split

Split patients into 4 groups (Q1-Q4) based on pathway score quartiles

In [ ]:
# Generate KM curves with quartile split for each significant pathway
for pathway in pathways_to_plot:
    print(f"\n{'='*80}")
    print(f"Pathway: {pathway}")
    print('='*80)
    
    try:
        wp_scanpy.plot_pathway_km_curves(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            cutpoint_method='quartiles',
            covariates=None,  # Quartile comparison typically doesn't use Cox adjustment
            figsize=(10, 6)
        )
    except Exception as e:
        print(f"Error plotting KM curve for {pathway}: {str(e)}")

### KM Curves: Optimal Cutpoint

Find optimal cutpoint (between 25th-75th percentile) that minimizes p-value.
Also display HR curve showing hazard ratios at each tested cutpoint.

In [ ]:
# Generate optimal cutpoint analysis for each significant pathway
# This includes both the HR curve and KM curve
for pathway in pathways_to_plot:
    print(f"\n{'='*80}")
    print(f"Pathway: {pathway}")
    print('='*80)
    
    try:
        # First, plot the HR curve showing all tested cutpoints
        print("\n--- HR Curve (showing optimal cutpoint) ---")
        cutpoint_results, optimal_cutpoint = wp_scanpy.plot_optimal_cutpoint_hr_curve(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex', 'Study_Site'],
            figsize=(10, 5)
        )
        
        # Then plot the KM curve using the optimal cutpoint
        print("\n--- KM Curve (using optimal cutpoint) ---")
        wp_scanpy.plot_pathway_km_curves(
            pdata_filt,
            pathway_name=pathway,
            ulm_obsm_key='score_ulm',
            cutpoint_method='optimal',
            covariates=['d_dx_amm_age', 'd_amm_tx_asct_1st', 'd_pt_sex', 'Study_Site'],
            figsize=(10, 6)
        )
        
    except Exception as e:
        print(f"Error in optimal cutpoint analysis for {pathway}: {str(e)}")